# Imports:

In [3]:

from itertools import permutations
import csv
import random
import math
import time
import copy


# 3D Container Packing – Explanation

## Overview

This environment simulates placing 3D boxes inside a container using a **smart placement rule (best-fit)**.  
It manages free space dynamically and evaluates packing quality using **utilization + stability**.

---

## Container Structure

The container keeps track of:

- `placed_boxes` → boxes successfully placed
- `unplaced_boxes` → boxes that couldn't fit
- `free_spaces` → available empty regions

Initially, the container has **one big free space** (the whole container).

---

## Box Model

Each box has:

- Dimensions: `l, w, h`
- `weight`
- `fragile` (limits rotation)

---

## Orientations

### `get_orientations(box)`

Generates possible rotations:

- **Fragile boxes** → only horizontal rotations  
- **Normal boxes** → all 6 permutations of (l, w, h)

👉 Purpose: maximize chances of fitting the box

---

## Fit Check

### `fits(l, w, h, space)`

Checks if a box orientation fits inside a free space:

l ≤ L, w ≤ W, h ≤ H

👉 Filters valid placements

---

## Placement Strategy (Smart Best-Fit)

### `find_best_fit_placement`

Core heuristic:

1. Try all orientations
2. Try all free spaces
3. Keep valid placements
4. Choose the best placement using minimum leftover, then tie-break by (z, x, y)

👉 Meaning:
- Pick the space with minimum leftover volume (best-fit)
- Prefer lower position first (gravity)
- Then leftmost
- Then front

---

## Placing a Box

### `place_box(box, choice)`

- If no valid position → add to `unplaced_boxes`
- Otherwise:
  - Place box at `(x, y, z)`
  - Remove used space
  - Split it into smaller spaces

---

## Space Splitting

### `split_space(...)`

After placing a box, the remaining space is split into:

- **Right** (remaining length)
- **Front** (remaining width)
- **Top** (remaining height)

👉 This keeps track of all empty regions dynamically

---

## Stability Evaluation

### `evaluate_stability()`

Returns 3 scores:

---

### 1. Weight Score

Checks if **heavier boxes are below lighter ones**

👉 Based on pairwise comparisons

---

### 2. Support Score

A box is stable if:

- It is on the ground  
- OR ≥ 30% of its base is supported by boxes below  

---

### 3. Final Stability

$$ \large stability = 0.4 × weight_score + 0.6 × support_score $$


👉 Support matters more than weight

---

## Packing Process

### `pack_boxes(boxes)`

Main pipeline:

1. Reset container
2. For each box:
   - Find best position using the smart placement rule (minimum leftover best-fit)
   - Place it
3. Compute:

- Number of placed/unplaced boxes
- Volume utilization
- Stability score

---

## Utilization
$$ \large utilization = packed_volume / container_volume $$

👉 Measures how well space is used

---

## Loading Boxes

### `load_random_boxes(csv_path, n, seed)`

- Reads boxes from CSV
- Randomly selects `n` boxes
- Converts them into usable format

---

## Key Idea

This system is:

- **Greedy** → fast but not optimal  
- **Geometry-based** → manages space via splitting  
- **Physics-aware** → considers stability  

👉 Perfect as a base for **GA / PSO optimization**


In [4]:

class Container:
    def __init__(self,length = 590,width = 235, height = 239):
        self.length = length
        self.width = width
        self.height = height
        self.placed_boxes = []
        self.free_spaces = [
            {"x":0,
             "y":0,
             "z":0,
             "L":length,
             "W":width,
             "H":height,
            }
        ]
        self.unplaced_boxes = []

        pass
    # returns all possible orientations of a box, example (30,40,20) =>  ( (30,40,20),(20,40,30) ect... )
    def get_orientations(self, box):

        l, w, h = box["l"], box["w"], box["h"]

        if box["fragile"]:
             return list(set([
            (l, w, h),
            (w, l, h)
            ]))

        return list(set(permutations((l, w, h))))

    #checks wether a given orientation of a box fits in a given space
    def fits(self,l,w,h,space):
         return l <= space["L"] and w <= space["W"] and h <= space["H"]


   
    
    def find_best_fit_placement(self, box):
        best_choice = None
        c = self
        for (l, w, h) in c.get_orientations(box):
            for space in c.free_spaces:
                if not c.fits(l, w, h, space):
                    continue
                leftover = (space["L"] * space["W"] * space["H"]) - (l * w * h)
                candidate_key = (leftover, space["z"], space["x"], space["y"])
                if best_choice is None or candidate_key < best_choice["key"]:
                    best_choice = {
                        "space": space,
                        "dims": (l, w, h),
                        "key": candidate_key,
                    }
        return best_choice

    
    #after finding where to place the box, we place it, then we remove the free space used by splitting it into smaller free spaces
    def place_box(self,box,choice):
        if choice is None:
            self.unplaced_boxes.append({
            "id": box["id"],
            "reason": "no valid placement found"
            })
            return False  # cannot place

        space = choice["space"]
        l, w, h = choice["dims"]

        x, y, z = space["x"], space["y"], space["z"]

        placed = {
        "id": box["id"],
        "name": box["name"],
        "weight": box["weight"],
        "x": x,
        "y": y,
        "z": z,
        "l": l,
        "w": w,
        "h": h
        }

        self.placed_boxes.append(placed)

    # remove used space
        self.free_spaces.remove(space)

    # split space
        new_spaces = self.split_space(space, l, w, h, x, y, z)

        self.free_spaces.extend(new_spaces)

        return True

    # we split the free space used to other to smaller free spaces
    def split_space(self, space, l, w, h, x, y, z):

        new_spaces = []

        # Right
        if space["L"] > l:
            new_spaces.append({
                "x": x + l,
                "y": y,
                "z": z,
                "L": space["L"] - l,
                "W": space["W"],
                "H": space["H"]
            })

        # Front
        if space["W"] > w:
            new_spaces.append({
                "x": x,
                "y": y + w,
                "z": z,
                "L": l,                 # only the box's footprint width, no overlap with Right
                "W": space["W"] - w,
                "H": space["H"]
            })

        # Top
        if space["H"] > h:
            new_spaces.append({
                "x": x,
                "y": y,
                "z": z + h,
                "L": l,                 # box footprint only, no overlap with Right or Front
                "W": w,
                "H": space["H"] - h
            })

        return new_spaces


    def evaluate_stability(self) -> dict:
    

        if not self.placed_boxes:
            return {"weight_score": 0.0, "support_score": 0.0, "stability_score": 0.0}

        # ── 1. WEIGHT DISTRIBUTION SCORE ────────────────────────────────────────
        # Idea: for every pair of boxes (A, B), if A is heavier than B,
        # A should sit lower (smaller z). We count how many pairs respect
        # this rule and divide by total pairs → fraction in [0, 1].

        pairs_correct = 0
        pairs_total   = 0

        for i, box_a in enumerate(self.placed_boxes):
            for box_b in self.placed_boxes[i + 1:]:

                w_a = box_a["weight"]
                w_b = box_b["weight"]

                # z is the bottom face of the box
                z_a = box_a["z"]
                z_b = box_b["z"]

                if w_a == w_b:
                    pairs_correct += 1      # equal weight: no violation
                elif w_a > w_b:
                    if z_a <= z_b:          # heavier A is at same level or below B ✓
                        pairs_correct += 1
                else:
                    if z_b <= z_a:          # heavier B is at same level or below A ✓
                        pairs_correct += 1

                pairs_total += 1

        weight_score = (pairs_correct / pairs_total * 100) if pairs_total > 0 else 100.0


        # ── 2. SUPPORT SCORE ────────────────────────────────────────────────────
        # A box is "supported" if it either:
        #   (a) sits on the floor  (z == 0), OR
        #   (b) has at least MIN_OVERLAP_RATIO of its base footprint
        #       covered by the top face of boxes directly below it.
        #
        # "Directly below" means: another box whose top face (z + h)
        # equals this box's bottom (z), and whose XY footprint overlaps.

        MIN_OVERLAP_RATIO = 0.30    # 30 % of base area must be supported

        supported_count = 0

        for box in self.placed_boxes:

            # boxes resting on the floor are always supported
            if box["z"] == 0:
                supported_count += 1
                continue

            box_x1, box_x2 = box["x"], box["x"] + box["l"]
            box_y1, box_y2 = box["y"], box["y"] + box["w"]
            box_base_area  = box["l"] * box["w"]

            supported_area = 0.0

            for other in self.placed_boxes:

                if other["id"] == box["id"]:
                    continue

                # other's top face must align with this box's bottom face
                other_top = other["z"] + other["h"]
                if other_top != box["z"]:
                    continue

                # compute XY overlap between footprints
                ox1, ox2 = other["x"], other["x"] + other["l"]
                oy1, oy2 = other["y"], other["y"] + other["w"]

                overlap_x = max(0, min(box_x2, ox2) - max(box_x1, ox1))
                overlap_y = max(0, min(box_y2, oy2) - max(box_y1, oy1))
                supported_area += overlap_x * overlap_y

            ratio = supported_area / box_base_area if box_base_area > 0 else 0

            if ratio >= MIN_OVERLAP_RATIO:
                supported_count += 1

        support_score = (supported_count / len(self.placed_boxes)) * 100


        # ── 3. COMBINED STABILITY SCORE ──────────────────────────────────────────
        # Weight distribution and support contribute equally by default.
        # You can tune these weights later when you build your fitness function.

        WEIGHT_IMPORTANCE  = 0.4
        SUPPORT_IMPORTANCE = 0.6    # support matters more: a floating box is dangerous

        stability_score = (
            weight_score  * WEIGHT_IMPORTANCE +
            support_score * SUPPORT_IMPORTANCE
        )

        return {
            "weight_score"   : round(weight_score,    2),
            "support_score"  : round(support_score,   2),
            "stability_score": round(stability_score, 2),
        } 




    # now this function will take a list of boxes with the order and try box by box to place it in the container, and returns the number of placed and unplaced boxes and show them 
    
    def pack_boxes(self, boxes): 
         self.placed_boxes = []
         self.unplaced_boxes = []
         self.free_spaces = [{
         "x": 0,
         "y": 0,
         "z": 0,
         "L": self.length,
         "W": self.width,
         "H": self.height
          }]

         for box in boxes:

            choice = self.find_best_fit_placement(box)

            self.place_box(box, choice)

        # statistics
         placed_count = len(self.placed_boxes)
         unplaced_count = len(self.unplaced_boxes)
         packed_volume = 0

         for box in self.placed_boxes:
            packed_volume += (
            box["l"] *
            box["w"] *
            box["h"]
            )

    # container volume
         container_volume = (
         self.length *
         self.width *
         self.height
         )

         utilization = (
         packed_volume / container_volume
         ) * 100
         stability = self.evaluate_stability()
         

         return {
        "placed": len(self.placed_boxes),
        "placed_count": len(self.placed_boxes),
        "unplaced": len(self.unplaced_boxes),
        "unplaced_count": len(self.unplaced_boxes),
        "utilization": utilization,
        "stability": stability,
        "placed_boxes": self.placed_boxes,
        "unplaced_boxes": self.unplaced_boxes
         }
        
    

        

def load_random_boxes(csv_path, n=100, seed=None):

    boxes = []

    if seed is not None:
        random.seed(seed)

    # open csv
    with open(csv_path, newline='', encoding='utf-8') as file:

        reader = csv.DictReader(file)

        # convert all rows to list
        rows = list(reader)

    # choose n random rows
    selected_rows = random.sample(rows, min(n, len(rows)))

    # convert rows to box dictionaries
    for row in selected_rows:

        box = {
            "id": row["id"],
            "name": row["name"],
            "l": int(row["length"]),
            "w": int(row["width"]),
            "h": int(row["height"]),
            "weight": float(row["weight"]),
            "fragile": row["fragile"].lower() == "true"
        }

        boxes.append(box)

    return boxes

# Simulated Annealing — Mathematical Explanation

## Objective Function (includes Placement Rate)

The goal is to evaluate the quality of a solution (i.e., a box ordering) using a weighted combination of three criteria:

- **Space Utilization**: how much of the container is filled
- **Stability**: how physically stable the packing is
- **Placement Rate**: placed boxes / total boxes * 100

---

### Formula

$$
\Large f(s) = w_{util} \cdot U(s) + w_{stab} \cdot S(s) + w_{space} \cdot P(s)
$$


#### Explanation

- $s$ → a solution (ordering of boxes)
- $U(s)$ → utilization score of solution $s$
- $S(s)$ → stability score of solution $s$
- $P(s)$ → placement rate of solution $s$ ($\frac{\text{placed boxes}}{\text{total boxes}} \times 100$)
- $w_{util}$ → weight of utilization (importance)
- $w_{stab}$ → weight of stability
- $w_{space}$ → weight of placement rate

---

### In this implementation

$$
w_{util} = 0.4, \quad w_{stab} = 0.3, \quad w_{space} = 0.3
$$

👉 This means:
- 40% importance is given to filling the container
- 30% importance is given to stability
- 30% importance is given to placement rate



### Acceptance Probability

Simulated Annealing allows accepting worse solutions to escape local optima.

---

### Difference Between Solutions

$$
\Delta = current\_score - new\_score
$$


#### Explanation

- If $\Delta < 0$ → new solution is better  
- If $\Delta > 0$ → new solution is worse  

---

### Acceptance Formula

$$
\Large P = e^{-\frac{\Delta}{T}}
$$



#### Explanation

- $P$ → probability of accepting a worse solution
- $\Delta$ → how much worse the new solution is
- $T$ → temperature (controls randomness)



#### Behavior

- If $T$ is **high**:
  - $P$ is high → more exploration
- If $T$ is **low**:
  - $P$ is low → more greedy behavior

---

### ✅ Decision Rule

$$
\text{Accept if } rand(0,1) < P
$$



#### Explanation

- Generate a random number between 0 and 1
- Accept the worse solution only if it is below $P$

---

## Cooling Schedule

The temperature decreases over time to reduce randomness.



#### Formula

$$
\Large T_{new} = T \cdot \alpha
$$



#### Explanation

- $T$ → current temperature
- $\alpha$ → cooling rate (between 0 and 1)

---

### Behavior

- If $\alpha$ is close to 1 → slow cooling (more exploration)
- If $\alpha$ is small → fast cooling (more greedy)

---

## Global Intuition

Simulated Annealing follows this logic:



### At High Temperature

- Accepts many solutions (even bad ones)
- Explores the search space widely



### At Low Temperature

- Rarely accepts worse solutions
- Focuses on improving current best solution




In [ ]:

class SimulatedAnnealing:
    def __init__(self,container,intitial_boxes,T=3500,alpha=0.97):
        self.container = container
        self.current = intitial_boxes
        self.T = T
        self.alpha = alpha

    def get_neighbor(self, solution):
        neighbor = solution.copy()

        i, j = random.sample(range(len(neighbor)), 2)
        neighbor[i], neighbor[j] = neighbor[j], neighbor[i]

        return neighbor
    
    def evaluate(self, solution):
       

        result = self.container.pack_boxes(solution)

        utilization = result["utilization"]
        stability = result["stability"]["stability_score"]
        placement = (result["placed"] /len(solution)) * 100

        w_util = 0.4
        w_stab = 0.3
        w_space = 0.3

        return w_util * utilization + w_stab * stability + w_space * placement
    

    def accept(self, current_score, new_score,T):
    
        # if new solution is better → always accept
        if new_score > current_score:
            return True
        T = max(T, 1e-10)
        # if worse → accept with probability
        delta = current_score - new_score
        probability = math.exp(-delta / T)

        return random.random() < probability
    

    def run(self,iterations=1500):

        # ── 1. INITIALIZATION ─────────────────────────────
        current_solution = self.current
        current_score = self.evaluate(current_solution)

        best_solution = current_solution
        best_score = current_score

        T = self.T   # initial temperature

        # ── 2. MAIN LOOP ───────────────────────────────────
        for _ in range(iterations):

            # generate neighbor
            neighbor = self.get_neighbor(current_solution)
            neighbor_score = self.evaluate(neighbor)

            # decide accept/reject
            if self.accept(current_score, neighbor_score,T):

                current_solution = neighbor
                current_score = neighbor_score

                # update best solution
                if neighbor_score > best_score:
                    best_solution = neighbor
                    best_score = neighbor_score

            # cool down
            T *= self.alpha
            

        # ── 3. RETURN RESULT ───────────────────────────────
        return {
            "best_solution": best_solution,
            "best_score": best_score,
            "best_result": self.container.pack_boxes(best_solution)  # final state
        }


In [6]:

if __name__ == "__main__":
    container = Container()
    boxes = load_random_boxes("../data/data.csv")
    SA = SimulatedAnnealing(container, boxes)
    sa_result = SA.run()
    result = sa_result["best_result"]

    print("========== SIMULATED ANNEALING ==========")
    print(f"Placed boxes   : {result['placed']}")
    print(f"Unplaced boxes : {result['unplaced']}")
    print(f"Score          : {sa_result['best_score']:.4f}")
    print()
    print(f"utilisation : {result['utilization']}")
    print(f"stabilty : {result['stability']}")

    print("Placed boxes:")
    for box in result["placed_boxes"]:
        print(box)

    print()

    print("Unplaced boxes:")
    for box in result["unplaced_boxes"]:
        print(box)


========== SIMULATED ANNEALING ==========
Placed boxes   : 75
Unplaced boxes : 25
Score          : 85.1466

utilisation : 84.13661019966895
stabilty : {'weight_score': 91.6, 'support_score': 100.0, 'stability_score': 96.64}
Placed boxes:
{'id': '255', 'name': 'Data Center Box', 'weight': 95.0, 'x': 0, 'y': 0, 'z': 0, 'l': 130, 'w': 160, 'h': 210}
{'id': '238', 'name': 'Production Equipment Box', 'weight': 110.0, 'x': 130, 'y': 0, 'z': 0, 'l': 200, 'w': 160, 'h': 170}
{'id': '24', 'name': 'Monitor Box', 'weight': 8.0, 'x': 130, 'y': 0, 'z': 170, 'l': 70, 'w': 20, 'h': 50}
{'id': '143', 'name': 'Pallet Jack Box', 'weight': 25.0, 'x': 200, 'y': 0, 'z': 170, 'l': 120, 'w': 80, 'h': 40}
{'id': '102', 'name': 'Desk Lamp Box', 'weight': 1.2, 'x': 200, 'y': 0, 'z': 210, 'l': 40, 'w': 25, 'h': 20}
{'id': '37', 'name': 'Heater Box', 'weight': 7.5, 'x': 200, 'y': 80, 'z': 170, 'l': 40, 'w': 60, 'h': 40}
{'id': '271', 'name': 'Large Assembly Box', 'weight': 70.0, 'x': 330, 'y': 0, 'z': 0, 'l': 150

# Particle Swarm Optimization (PSO)

## Overview

This implementation uses **Particle Swarm Optimization (PSO)** to optimize  
the **ordering of boxes** for container packing.

Each particle represents a **candidate solution** (ordering of boxes),  
and the swarm collaboratively searches for the best arrangement.

---

# Objective Function

The quality of a solution is measured using:

$$
\Large f(s) = U(s) + \lambda_{stab} \cdot S(s) - \lambda_{unplaced} \cdot N_{unplaced}(s)
$$



#### Explanation

- $s$ → a solution (ordering of boxes)
- $U(s)$ → utilization (0–100)
- $S(s)$ → stability score (0–100)
- $N_{unplaced}(s)$ → number of boxes not placed
- $\lambda_{stab}$ → weight of stability
- $\lambda_{unplaced}$ → penalty for unplaced boxes

---

## Interpretation

- Maximize space usage → $U(s)$
- Favor stable packing → $S(s)$
- Penalize failures → $N_{unplaced}(s)$

---

## Helper Function

### `_evaluate_order(...)`

#### Purpose

Evaluates a solution by:

1. Packing boxes into the container
2. Extracting metrics
3. Computing fitness



### Returned Values

- `fitness` → scalar score
- `stats` → detailed metrics:
  - utilization
  - stability
  - placed/unplaced boxes

---

## Particle Representation

Each particle encodes a solution using **continuous values**.



### Position

$$
\mathbf{x}_i = (x_{i1}, x_{i2}, ..., x_{in}), \quad x_{ij} \in [0,1]
$$



#### Meaning

- Each dimension corresponds to a box
- Sorting positions → gives box order

---

## Velocity

$$
\mathbf{v}_i = (v_{i1}, v_{i2}, ..., v_{in})
$$

- Controls how the position changes over time

---

## Personal Best

Each particle keeps:

$$
\mathbf{p}_i = \text{best position found by particle } i
$$



## Global Best

The swarm tracks:

$$
\mathbf{g} = \text{best position found by all particles}
$$

---

## Decoding Function

### `decode_order(boxes)`

#### Idea

Convert continuous position → discrete permutation



#### Method

- Sort indices by position values
- Reorder boxes accordingly

---

## Velocity Update

### `update_velocity(...)`



#### Formula

$$
\Large v_{ij}(t+1) =
w \cdot v_{ij}(t)
+ c_1 r_1 (p_{ij} - x_{ij})
+ c_2 r_2 (g_j - x_{ij})
$$



#### Explanation

- $v_{ij}$ → velocity of particle $i$ in dimension $j$
- $x_{ij}$ → position
- $p_{ij}$ → personal best
- $g_j$ → global best
- $w$ → inertia weight
- $c_1$ → cognitive coefficient
- $c_2$ → social coefficient
- $r_1, r_2 \sim U(0,1)$ → random numbers



### Interpretation

- **Inertia term** ($w \cdot v$):
  - keeps current movement direction

- **Cognitive term**:
  - pulls particle toward its own best

- **Social term**:
  - pulls particle toward global best

---

## Position Update

### `update_position()`



### Formula

$$
\Large x_{ij}(t+1) = x_{ij}(t) + v_{ij}(t+1)
$$




### Constraint

$$
\Large x_{ij} \in [0,1]
$$



####  Explanation

- After update, values are **clamped**:
  
$$
x_{ij} = \max(0, \min(1, x_{ij}))
$$

- Ensures valid ordering behavior

---

##  PSO Parameters



### . Inertia Weight

- $w \in [0.4, 0.9]$

👉 Controls exploration vs exploitation


### . Cognitive Coefficient

- $c_1 \approx 1.5$

👉 Trust in personal experience


### . Social Coefficient

- $c_2 \approx 1.5$

👉 Trust in swarm knowledge


### . Fitness Weights

- $\lambda_{stab}$ → importance of stability
- $\lambda_{unplaced}$ → penalty strength

---

## Main Algorithm

### `run()`



### 1️⃣ Initialization

- Create swarm of particles
- Initialize:
  - positions
  - velocities
- Set:

$$
\mathbf{g} = \text{initial global best}
$$



### 2️⃣ Iterative Optimization

For each iteration:



#### Step 1: Evaluate Particles

For each particle:

$$
f_i = f(s_i)
$$



#### Step 2: Update Personal Best

$$
\text{if } f_i > f(p_i) \Rightarrow p_i = x_i
$$



#### Step 3: Update Global Best

$$
\text{if } f_i > f(g) \Rightarrow g = x_i
$$



#### Step 4: Update Velocity

Using:

$$
v_{ij}(t+1) =
w v_{ij}(t)
+ c_1 r_1 (p_{ij} - x_{ij})
+ c_2 r_2 (g_j - x_{ij})
$$



#### Step 5: Update Position

$$
x_{ij}(t+1) = x_{ij}(t) + v_{ij}(t+1)
$$



### 3️⃣ Final Solution

- Select best particle:

$$
\arg\max_i f(p_i)
$$

- Decode to box order
- Re-pack container

---

## Convergence Tracking

At each iteration:

Store:

- best fitness
- utilization
- stability
- placed/unplaced counts

---

## Execution Time

Measured as:

$$
\text{elapsed} = t_{end} - t_{start}
$$

---


## Intuition

- Particles explore different box orders
- Good solutions spread through the swarm
- Balance between:
  - exploration (randomness)
  - exploitation (best solutions)

👉 Result: progressively better packing configurations

In [7]:


# ─────────────────────────────────────────────────────────────────────────────
# Helper: thin wrapper so we don't mutate the caller's Container instance
# ─────────────────────────────────────────────────────────────────────────────

def _evaluate_order(container, boxes_in_order,
                    lambda_stability, lambda_unplaced):
    """
    Pack *boxes_in_order* into a fresh container state and return
    (fitness, stats_dict).
    """
    result = container.pack_boxes(boxes_in_order)

    utilization    = result["utilization"]                        # 0-100
    stability      = result["stability"]["stability_score"]       # 0-100
    unplaced_count = result["unplaced_count"]

    fitness = (
        utilization
        + lambda_stability * stability
        - lambda_unplaced  * unplaced_count
    )

    stats = {
        "fitness"       : round(fitness,       4),
        "utilization"   : round(utilization,   4),
        "stability"     : round(stability,     4),
        "placed_count"  : result["placed_count"],
        "unplaced_count": unplaced_count,
        "placed_boxes"  : result["placed_boxes"],
        "unplaced_boxes": result["unplaced_boxes"],
    }

    return fitness, stats


# ─────────────────────────────────────────────────────────────────────────────
# Particle
# ─────────────────────────────────────────────────────────────────────────────

class Particle:
    """One particle in the swarm."""

    def __init__(self, n_boxes, rng):
        # position: random floats in [0, 1], one per box
        self.position = [rng.random() for _ in range(n_boxes)]
        # velocity: small random floats
        self.velocity = [rng.uniform(-0.1, 0.1) for _ in range(n_boxes)]

        self.best_position = list(self.position)
        self.best_fitness  = float("-inf")
        self.current_stats = None

    # ── decode ──────────────────────────────────────────────────────────────
    def decode_order(self, boxes):
        """
        Sort boxes by their position values to get a packing order.
        Returns a new list of boxes (does NOT mutate the original).
        """
        indexed = sorted(enumerate(self.position), key=lambda x: x[1])
        return [boxes[i] for i, _ in indexed]

    # ── update ──────────────────────────────────────────────────────────────
    def update_velocity(self, global_best_pos, w, c1, c2, rng):
        n = len(self.position)
        for i in range(n):
            r1 = rng.random()
            r2 = rng.random()

            cognitive = c1 * r1 * (self.best_position[i] - self.position[i])
            social    = c2 * r2 * (global_best_pos[i]    - self.position[i])

            self.velocity[i] = (
                w * self.velocity[i]
                + cognitive
                + social
            )

    def update_position(self):
        n = len(self.position)
        for i in range(n):
            self.position[i] += self.velocity[i]
            # clamp to [0, 1] to keep rank-ordering numerically stable
            self.position[i] = max(0.0, min(1.0, self.position[i]))


# ─────────────────────────────────────────────────────────────────────────────
# PSOPacker
# ─────────────────────────────────────────────────────────────────────────────

class PSOPacker:
    """
    Parameters
    ----------
    container        : Container  – your existing Container instance
    boxes            : list[dict] – full box list (from load_random_boxes)
    n_particles      : int        – swarm size (20-50 is typical)
    n_iter           : int        – number of iterations
    w                : float      – inertia weight (0.4-0.9)
    c1               : float      – cognitive coefficient (≈1.5)
    c2               : float      – social coefficient   (≈1.5)
    lambda_stability : float      – weight of stability in fitness
    lambda_unplaced  : float      – penalty per unplaced box
    seed             : int|None   – random seed for reproducibility
    verbose          : bool       – print progress every iteration
    """

    def __init__(
        self,
        container,
        boxes,
        n_particles      = 30,
        n_iter           = 100,
        w                = 0.7,
        c1               = 1.5,
        c2               = 1.5,
        lambda_stability = 0.3,
        lambda_unplaced  = 2.0,
        seed             = None,
        verbose          = True,
    ):
        self.container        = container
        self.boxes            = boxes
        self.n_particles      = n_particles
        self.n_iter           = n_iter
        self.w                = w
        self.c1               = c1
        self.c2               = c2
        self.lambda_stability = lambda_stability
        self.lambda_unplaced  = lambda_unplaced
        self.verbose          = verbose

        self.rng = random.Random(seed)

        self.n_boxes = len(boxes)

        # swarm
        self.particles = [
            Particle(self.n_boxes, self.rng)
            for _ in range(n_particles)
        ]

        # global best
        self.global_best_position = list(self.particles[0].position)
        self.global_best_fitness  = float("-inf")
        self.global_best_stats    = None

        # convergence history (one entry per iteration)
        self.history = []

    # ── main loop ────────────────────────────────────────────────────────────

    def run(self):
        t0 = time.time()

        for iteration in range(1, self.n_iter + 1):

            # ── evaluate every particle ──────────────────────────────────────
            for particle in self.particles:

                order   = particle.decode_order(self.boxes)
                fitness, stats = _evaluate_order(
                    self.container, order,
                    self.lambda_stability, self.lambda_unplaced
                )
                particle.current_stats = stats

                # update personal best
                if fitness > particle.best_fitness:
                    particle.best_fitness  = fitness
                    particle.best_position = list(particle.position)

                # update global best
                if fitness > self.global_best_fitness:
                    self.global_best_fitness  = fitness
                    self.global_best_position = list(particle.position)
                    self.global_best_stats    = copy.deepcopy(stats)

            # ── log ──────────────────────────────────────────────────────────
            self.history.append({
                "iteration"     : iteration,
                "best_fitness"  : round(self.global_best_fitness, 4),
                "utilization"   : self.global_best_stats["utilization"],
                "stability"     : self.global_best_stats["stability"],
                "placed_count"  : self.global_best_stats["placed_count"],
                "unplaced_count": self.global_best_stats["unplaced_count"],
            })

            if self.verbose:
                h = self.history[-1]
                print(
                    f"[PSO] iter {iteration:>4}/{self.n_iter}  "
                    f"fitness={h['best_fitness']:>8.3f}  "
                    f"util={h['utilization']:>6.2f}%  "
                    f"stab={h['stability']:>6.2f}  "
                    f"placed={h['placed_count']}/{self.n_boxes}"
                )

            # ── update velocities & positions ─────────────────────────────────
            for particle in self.particles:
                particle.update_velocity(
                    self.global_best_position,
                    self.w, self.c1, self.c2, self.rng
                )
                particle.update_position()

        # ── rebuild best layout one final time ───────────────────────────────
        best_particle = max(self.particles, key=lambda p: p.best_fitness)
        best_order    = best_particle.decode_order(self.boxes)

        # re-pack so container.placed_boxes reflects the best solution
        self.container.pack_boxes(best_order)

        elapsed = time.time() - t0

        return {
            "best_order"    : best_order,
            "stats"         : self.global_best_stats,
            "history"       : self.history,
            "elapsed_sec"   : round(elapsed, 2),
        }


In [8]:


# ─────────────────────────────────────────────────────────────────────────────
# Quick self-test  (runs only when executed directly)
# ─────────────────────────────────────────────────────────────────────────────

if __name__ == "__main__":

    # ── minimal inline box list so the file is self-contained ────────────────
    sample_boxes = load_random_boxes("../data/data.csv")

    
    container = Container()

    pso = PSOPacker(
        container        = container,
        boxes            = sample_boxes,
        n_particles      = 100,
        n_iter           = 50,
        w                = 0.7,
        c1               = 1.5,
        c2               = 1.5,
        lambda_stability = 0.3,
        lambda_unplaced  = 5.0,
        seed             = 42,
        verbose          = True,
    )

    result = pso.run()

    print("\n=== PSO RESULT ===")
    print(f"  Elapsed      : {result['elapsed_sec']}s")
    print(f"  Utilization  : {result['stats']['utilization']}%")
    print(f"  Stability    : {result['stats']['stability']}")
    print(f"  Placed       : {result['stats']['placed_count']} / {len(sample_boxes)}")
    print(f"  Unplaced     : {result['stats']['unplaced_count']}")


[PSO] iter    1/50  fitness=   5.294  util= 72.27%  stab= 93.41  placed=81/100
[PSO] iter    2/50  fitness=  15.574  util= 72.65%  stab= 93.08  placed=83/100
[PSO] iter    3/50  fitness=  20.827  util= 77.72%  stab= 93.69  placed=83/100
[PSO] iter    4/50  fitness=  20.827  util= 77.72%  stab= 93.69  placed=83/100
[PSO] iter    5/50  fitness=  21.367  util= 78.30%  stab= 93.56  placed=83/100
[PSO] iter    6/50  fitness=  22.810  util= 79.60%  stab= 94.04  placed=83/100
[PSO] iter    7/50  fitness=  23.128  util= 79.60%  stab= 95.10  placed=83/100
[PSO] iter    8/50  fitness=  23.128  util= 79.60%  stab= 95.10  placed=83/100
[PSO] iter    9/50  fitness=  23.131  util= 79.60%  stab= 95.11  placed=83/100
[PSO] iter   10/50  fitness=  23.131  util= 79.60%  stab= 95.11  placed=83/100
[PSO] iter   11/50  fitness=  23.131  util= 79.60%  stab= 95.11  placed=83/100
[PSO] iter   12/50  fitness=  23.131  util= 79.60%  stab= 95.11  placed=83/100
[PSO] iter   13/50  fitness=  23.131  util= 79.60%  

# Greedy Best-Fit Decreasing (BFD)

## Overview

This algorithm packs boxes using a **greedy strategy**:

1. Sort boxes by **decreasing volume**
2. For each box, choose the **best-fitting space**

---

## Sorting Strategy

Boxes are sorted by volume:

$$
\Large V = l \cdot w \cdot h
$$



### Meaning

- Larger boxes are placed first
- Reduces risk of leaving unusable large gaps

---

##  Best-Fit Placement

### `_find_best_fit_placement(box)`

For each:

- orientation $(l,w,h)$  
- free space  

Check feasibility:

$$
l \leq L,\quad w \leq W,\quad h \leq H
$$

---

## Selection Criterion

Choose placement minimizing:

$$
\Large \text{key} = (\text{leftover}, z, x, y)
$$



### Where

$$
\text{leftover} = (L \cdot W \cdot H) - (l \cdot w \cdot h)
$$



### Interpretation

- Minimize unused space
- Prefer lower ($z$), then left ($x$), then front ($y$)

---

## Packing Loop

For each sorted box:

- Find best placement
- Place using container logic

---

## Utilization

After packing:

$$
\Large \text{Utilization} = \frac{\text{Packed Volume}}{\text{Container Volume}} \times 100
$$



#### 📌 Where

$$
\text{Packed Volume} = \sum (l \cdot w \cdot h)
$$

$$
\text{Container Volume} = L \cdot W \cdot H
$$

---

## Stability

Obtained from container:

$$
\text{Stability} = S(s)
$$

---

## Final Score

The score also incorporates the placement rate, defined as the number of placed boxes divided by the total number of boxes, multiplied by 100.

$$
\Large \text{Score} = 0.4 \cdot \text{Utilization} + 0.3 \cdot \text{Stability} + 0.3 \cdot \text{PlacementRate}
$$

---

## Output

Returns:

- best solution (sorted boxes)
- score
- stats:
  - placed / unplaced
  - utilization
  - stability

---

## Intuition

- Pack big boxes first

- Always choose the tightest fit- Fast but **not globally optimal**

In [9]:
class GreedyBFD:
    """Greedy Best-Fit Decreasing on the shared Container: volume-decreasing order, best-fit placement."""

    def __init__(self, container, initial_boxes):
        self.container = container
        self.boxes = initial_boxes

    def _find_best_fit_placement(self, box):
        best_choice = None
        c = self.container
        for (l, w, h) in c.get_orientations(box):
            for space in c.free_spaces:
                if not c.fits(l, w, h, space):
                    continue
                leftover = (space["L"] * space["W"] * space["H"]) - (l * w * h)
                candidate_key = (leftover, space["z"], space["x"], space["y"])
                if best_choice is None or candidate_key < best_choice["key"]:
                    best_choice = {
                        "space": space,
                        "dims": (l, w, h),
                        "key": candidate_key,
                    }
        return best_choice

    def run(self):
        c = self.container
        sorted_boxes = sorted(
            self.boxes, key=lambda b: b["l"] * b["w"] * b["h"], reverse=True
        )
        c.placed_boxes = []
        c.unplaced_boxes = []
        c.free_spaces = [
            {
                "x": 0,
                "y": 0,
                "z": 0,
                "L": c.length,
                "W": c.width,
                "H": c.height,
            }
        ]
        for box in sorted_boxes:
            c.place_box(box, self._find_best_fit_placement(box))
        packed_volume = sum(b["l"] * b["w"] * b["h"] for b in c.placed_boxes)
        cv = c.length * c.width * c.height
        utilization = (packed_volume / cv) * 100 if cv else 0.0
        stability = c.evaluate_stability()
        best_result = {
            "placed": len(c.placed_boxes),
            "unplaced": len(c.unplaced_boxes),
            "utilization": utilization,
            "stability": stability,
            "placed_boxes": c.placed_boxes,
            "unplaced_boxes": c.unplaced_boxes,
        }
        w_util, w_space,w_stab = 0.4, 0.3,0.3
        best_score = w_util * utilization + w_stab * stability["stability_score"] +( w_space * (best_result["placed"] / len(sorted_boxes)) * 100 )
        return {
            "best_solution": sorted_boxes,
            "best_score": best_score,
            "best_result": best_result,
        }


In [10]:
if __name__ == "__main__":
    container = Container()
    boxes = load_random_boxes("../data/data.csv")

    bfd = GreedyBFD(container, boxes)
    bfd_result = bfd.run()
    r_bfd = bfd_result["best_result"]
    print("========== GREEDY BFD (same container) ==========")
    print(f"Placed boxes   : {r_bfd['placed']}")
    print(f"Unplaced boxes : {r_bfd['unplaced']}")
    print(f"Score          : {bfd_result['best_score']:.4f}")
    print(f"Utilization    : {r_bfd['utilization']}")
    print(f"Stability      : {r_bfd['stability']}")
    print()
    print("Placed boxes:")
    for box in r_bfd["placed_boxes"]:
        print(box)

    print()

    print("Unplaced boxes:")
    for box in r_bfd["unplaced_boxes"]:
        print(box)


========== GREEDY BFD (same container) ==========
Placed boxes   : 49
Unplaced boxes : 51
Score          : 76.9214
Utilization    : 86.01341084908721
Stability      : {'weight_score': 81.8, 'support_score': 100.0, 'stability_score': 92.72}

Placed boxes:
{'id': '248', 'name': 'Transport Crate XL', 'weight': 190.0, 'x': 0, 'y': 0, 'z': 0, 'l': 270, 'w': 230, 'h': 210}
{'id': '247', 'name': 'Massive Equipment Box', 'weight': 180.0, 'x': 270, 'y': 0, 'z': 0, 'l': 260, 'w': 200, 'h': 220}
{'id': '174', 'name': 'Robot Arm Box', 'weight': 40.0, 'x': 530, 'y': 0, 'z': 0, 'l': 60, 'w': 120, 'h': 80}
{'id': '144', 'name': 'Forklift Parts Box', 'weight': 40.0, 'x': 530, 'y': 0, 'z': 80, 'l': 60, 'w': 100, 'h': 80}
{'id': '166', 'name': 'Stage Equipment Box', 'weight': 35.0, 'x': 530, 'y': 120, 'z': 0, 'l': 50, 'w': 80, 'h': 120}
{'id': '207', 'name': 'Heavy Tools Box', 'weight': 28.0, 'x': 530, 'y': 0, 'z': 160, 'l': 60, 'w': 95, 'h': 70}
{'id': '27', 'name': 'TV 65 inch Box', 'weight': 24.0, 'x

# 🧬 Genetic Algorithm (GA)

## Overview

This algorithm optimizes the **ordering of boxes** using evolutionary principles:

- Selection
- Crossover
- Mutation
- Elitism
- Restart (to avoid stagnation)

---

## Objective Function (includes Placement Rate)

$$
\Large f(s) = w_{util} \cdot U(s) + w_{stab} \cdot S(s) + w_{space} \cdot P(s)
$$



#### 📌 Explanation

- $s$ → chromosome (box ordering)
- $U(s)$ → utilization (0–100)
- $S(s)$ → stability score (0–100)
- $P(s)$ → placement rate (placed boxes / total boxes * 100)
- $w_{util} = 0.4$
- $w_{stab} = 0.3$
- $w_{space} = 0.3$

---

## Chromosome Representation

A chromosome is:

$$
\Large s = (b_1, b_2, ..., b_n)
$$

- A **permutation of boxes**
- Order determines packing sequence

---

## Auto-Scaled Parameters



#### . Population Size

$$
\Large pop = \min(200,\ \max(50,\ n))
$$



#### . Generations

$$
\Large gen = \max\left(100,\ \frac{max\_evals}{pop}\right)
$$



#### . Mutation Rate

$$
\Large p_m = \min\left(0.30,\ \frac{2}{\sqrt{n}}\right)
$$



#### . Tournament Size

$$
\Large k = \max(3,\ 0.05 \cdot pop)
$$



#### . Elitism Count

$$
\Large elite = \max(2,\ 0.03 \cdot pop)
$$



#### . Stagnation Limit

$$
\Large stagnation = \max(30,\ \frac{5}{p_m})
$$

---

### Interpretation

- Larger problems → adaptive parameters
- Keeps balance between:
  - exploration
  - convergence speed

---

## Initialization

### `_init_population()`

- Generate random permutations:

$$
\text{population} = \{ \text{random permutations of } s \}
$$

---

## Selection

### `_tournament_select()`



#### Process

- Pick $k$ random individuals
- Select the best:

$$
\Large parent = \arg\max f(s_i)
$$

---

##  Crossover

### `_ox_crossover(p1, p2)`

**Order Crossover (OX)**


##### Steps

1. Select segment $[a,b]$
2. Copy segment from parent
3. Fill remaining positions in order



### Property

- Produces valid permutations
- Preserves relative ordering

---

## Mutation

### `_swap_mutate()`



#### Operation

Perform two swaps:

$$
(b_i, b_j) \leftrightarrow (b_j, b_i)
$$



#### Purpose

- Introduces diversity
- Avoids premature convergence

---

## Restart Mechanism

### `_restart()`



#### Idea

If no improvement:

- Keep top 50%
- Replace rest with random solutions



#### Condition

$$
\text{if stagnation} \geq stagnation\_limit
$$

---

## Main Algorithm

### `run()`



### 1 Initialization

- Generate population
- Compute fitness

$$
f_i = f(s_i)
$$



### 2️ Evolution Loop

For each generation:


---


#### 🏆 Elitism

Keep best individuals:

$$
\text{Top } elite \text{ survive}
$$



#### 🎯 Selection

Select parents via tournament



#### 🔀 Crossover

With probability:

$$
\Large P_c = 0.85
$$



#### 🔁 Mutation

With probability:

$$
\Large P_m = mutation\_rate
$$



#### 📊 Evaluation

$$
f_i = f(s_i)
$$



#### 🏆 Update Best

$$
\text{if } f(s_i) > f_{best} \Rightarrow update
$$



#### 🔁 Restart (if needed)

$$
\text{if stagnation} \geq limit \Rightarrow restart
$$

---

### 3️⃣ Final Output

Return:

- best solution
- best score
- packing result
- convergence history

---


## Intuition

- Population evolves toward better packings
- Good solutions survive and combine
- Mutation + restart prevent getting stuck
- Balances:

  - exploitation (elitism)  - exploration (mutation, restart)

In [11]:

class GeneticAlgorithm:
    def __init__(
        self,
        container,
        boxes,
        max_evals=40_000,     # ← ONLY TUNE THIS — controls total runtime
        pop_size=None,
        generations=None,
        crossover_rate=0.85,
        mutation_rate=None,
        tournament_k=None,
        elitism_count=None,
        w_util=0.4,
        w_space = 0.3,
        w_stab=0.3,
        stagnation_limit=None,
    ):
        self.container      = container
        self.boxes          = boxes
        self.crossover_rate = crossover_rate
        self.w_util         = w_util
        self.w_stab         = w_stab
        self.history        = []
        self.w_space        = w_space

        n = len(boxes)

        # ── AUTO-SCALE FROM n AND max_evals ──────────────────────────
        self.pop_size         = pop_size      or min(200, max(50, n))
        self.generations      = generations   or max(100, max_evals // self.pop_size)
        self.mutation_rate    = mutation_rate or round(min(0.30, 2 / math.sqrt(n)), 3)
        self.tournament_k     = tournament_k  or max(3, round(0.05 * self.pop_size))
        self.elitism_count    = elitism_count or max(2, round(0.03 * self.pop_size))
        self.stagnation_limit = stagnation_limit or max(30, round(5 / self.mutation_rate))

        print(f"── GA Parameters (n={n} boxes) ─────────────────────")
        print(f"  pop_size         = {self.pop_size}")
        print(f"  generations      = {self.generations}")
        print(f"  crossover_rate   = {self.crossover_rate}")
        print(f"  mutation_rate    = {self.mutation_rate}")
        print(f"  tournament_k     = {self.tournament_k}")
        print(f"  elitism_count    = {self.elitism_count}")
        print(f"  stagnation_limit = {self.stagnation_limit}")
        print(f"  total evals      = {self.pop_size * self.generations:,}")
        print("─────────────────────────────────────────────────────")

    def evaluate(self, chromosome):
        result = self.container.pack_boxes(chromosome)
        return (self.w_util * result["utilization"]
                + self.w_stab * result["stability"]["stability_score"] + self.w_space * (result["placed"] / len(chromosome)) * 100)

    def _init_population(self):
        population = []
        for _ in range(self.pop_size):
            chrom = self.boxes[:]
            random.shuffle(chrom)
            population.append(chrom)
        return population

    def _tournament_select(self, population, fitnesses):
        def one():
            contestants = random.sample(range(len(population)), self.tournament_k)
            return population[max(contestants, key=lambda i: fitnesses[i])]
        return one(), one()

    def _ox_crossover(self, p1, p2):
        n    = len(p1)
        a, b = sorted(random.sample(range(n), 2))

        def build(main, other):
            seg_ids      = {box["id"] for box in main[a:b+1]}
            child        = [None] * n
            child[a:b+1] = main[a:b+1]
            remainder    = [box for box in other if box["id"] not in seg_ids]
            for pos, box in zip([i for i in range(n) if child[i] is None], remainder):
                child[pos] = box
            return child

        return build(p1, p2), build(p2, p1)

    def _swap_mutate(self, chromosome):
        mutant = chromosome[:]
        # double swap for stronger perturbation
        i, j = random.sample(range(len(mutant)), 2)
        mutant[i], mutant[j] = mutant[j], mutant[i]
        i, j = random.sample(range(len(mutant)), 2)
        mutant[i], mutant[j] = mutant[j], mutant[i]
        return mutant

    def _restart(self, population, fitnesses):
        sorted_idx = sorted(range(len(population)),
                            key=lambda i: fitnesses[i], reverse=True)
        survivors  = [population[i][:] for i in sorted_idx[:self.pop_size // 2]]
        fresh = []
        for _ in range(self.pop_size - len(survivors)):
            chrom = self.boxes[:]
            random.shuffle(chrom)
            fresh.append(chrom)
        return survivors + fresh

    def run(self):
        population = self._init_population()
        fitnesses  = [self.evaluate(c) for c in population]

        best_idx      = max(range(len(population)), key=lambda i: fitnesses[i])
        best_solution = population[best_idx][:]
        best_score    = fitnesses[best_idx]
        self.history.append(best_score)
        stagnation_counter = 0
        print(f"Gen 000 | best score: {best_score:.4f}")

        for gen in range(1, self.generations + 1):

            sorted_idx     = sorted(range(len(population)),
                                    key=lambda i: fitnesses[i], reverse=True)
            new_population = [population[i][:] for i in sorted_idx[:self.elitism_count]]

            while len(new_population) < self.pop_size:
                p1, p2 = self._tournament_select(population, fitnesses)
                if random.random() < self.crossover_rate:
                    c1, c2 = self._ox_crossover(p1, p2)
                else:
                    c1, c2 = p1[:], p2[:]
                if random.random() < self.mutation_rate:
                    c1 = self._swap_mutate(c1)
                if random.random() < self.mutation_rate:
                    c2 = self._swap_mutate(c2)
                new_population.append(c1)
                if len(new_population) < self.pop_size:
                    new_population.append(c2)

            population = new_population
            fitnesses  = [self.evaluate(c) for c in population]

            gen_best = max(range(len(population)), key=lambda i: fitnesses[i])
            if fitnesses[gen_best] > best_score:
                best_score         = fitnesses[gen_best]
                best_solution      = population[gen_best][:]
                stagnation_counter = 0
            else:
                stagnation_counter += 1

            if stagnation_counter >= self.stagnation_limit:
                population         = self._restart(population, fitnesses)
                fitnesses          = [self.evaluate(c) for c in population]
                stagnation_counter = 0
                print(f"Gen {gen:03d} | *** RESTART ***")

            self.history.append(best_score)
            if gen % 20 == 0:
                print(f"Gen {gen:03d} | best score: {best_score:.4f}")

        print(f"Gen {self.generations:03d} | FINAL best score: {best_score:.4f}")

        return {
            "best_solution": best_solution,
            "best_score":    best_score,
            "best_result":   self.container.pack_boxes(best_solution),
            "history":       self.history,
        }


In [12]:
if __name__ == "__main__":
    container = Container()
    boxes     = load_random_boxes("../data/data.csv", n=200)  # ← change n freely

    ga = GeneticAlgorithm(
        container=container,
        boxes=boxes,
        max_evals=1_000,    # ← tune this for runtime: 40k ≈ 10 min for n=200
    )

    ga_result = ga.run()
    result    = ga_result["best_result"]

    print()
    print("========== GA PACKING RESULTS ==========")
    print(f"Best score     : {ga_result['best_score']:.4f}")
    print(f"Placed boxes   : {result['placed']}")
    print(f"Unplaced boxes : {result['unplaced']}")
    print()
    print(f"Utilisation    : {result['utilization']:.2f} %")
    print(f"Stability      : {result['stability']}")
    print()
    print("Placed boxes:")
    for box in result["placed_boxes"]:
        print(box)
    print()
    print("Unplaced boxes:")
    for box in result["unplaced_boxes"]:
        print(box)

── GA Parameters (n=200 boxes) ─────────────────────
  pop_size         = 200
  generations      = 100
  crossover_rate   = 0.85
  mutation_rate    = 0.141
  tournament_k     = 10
  elitism_count    = 6
  stagnation_limit = 35
  total evals      = 20,000
─────────────────────────────────────────────────────
Gen 000 | best score: 80.3404


KeyboardInterrupt: 

# Ant Colony Optimization


### Core Idea
Real ants find short paths by depositing pheromone — shorter paths get traversed more, accumulate more pheromone, and attract more ants. ACO copies this: artificial ants explore box orderings, reinforce good ones with pheromone, and the colony converges toward high-utilization packings.

The search space is all permutations of n boxes (n! possibilities). ACO navigates it by learning which box-to-box transitions lead to good results.

### Graph Model
```
Nodes : one per box  +  one virtual START node
Edges : (i → j) = "box j is placed immediately after box i"
τ[i][j] : pheromone on that edge  (learned)
η[j]    : heuristic desirability of box j  (fixed)
```
Each ant builds a **Hamiltonian path** through all n boxes. That path = a packing order fed into `pack_boxes()`.

### Selection Formula
At each step, an ant at node `i` picks the next unvisited box `j` with:

$$P(j \mid i) = \frac{\tau_{ij}^{\alpha} \cdot \eta_j^{\beta}}{\sum_{k \in \text{unvisited}} \tau_{ik}^{\alpha} \cdot \eta_k^{\beta}}$$

High pheromone (`τ`) = historically good transition. High heuristic (`η`) = physically good candidate (large, heavy, non-fragile).

### Heuristic η
```python
η[j] = sqrt(volume × weight) × (0.5 if fragile else 1.0)
```
Large heavy boxes get high η → placed early (bottom layers). Fragile boxes get η halved → deferred to upper layers.

### Pheromone Update (per iteration)
```
# 1. Evaporate all edges
τ[i][j]  ←  (1 - ρ) × τ[i][j]

# 2. Each ant deposits along its tour
Δτ = Q / score_ant
τ[i][j]  ←  τ[i][j]  +  Δτ

# 3. Clamp to [tau_min, tau_max]
```
Better ants deposit more. Evaporation prevents stagnation. `tau_min` keeps all edges explorable forever.

### Parameters

| Parameter | Default | Role |
|---|---|---|
| `n_ants` | 30 | Solutions built per iteration |
| `n_iter` | 100 | Total iterations |
| `alpha` | 1.0 | Pheromone trust (higher → exploit) |
| `beta` | 2.0 | Heuristic trust (higher → greedy) |
| `rho` | 0.1 | Evaporation rate — higher = more exploration |
| `Q` | 100.0 | Pheromone deposit scale |
| `tau_min` | 0.01 | Floor — prevents permanent edge exclusion |
| `tau_max` | None | Ceiling — prevents monopoly on one solution |

### Usage
```python
from container   import Container
from aco_packing import ACOPacker

aco = ACOPacker(
    container=container, boxes=boxes,
    n_ants=30, n_iter=100,
    alpha=1.0, beta=2.0, rho=0.1, Q=100.0,
    seed=42, verbose=True,
)
result = aco.run()

# result["stats"] keys:
#   utilization, stability, placed_count, unplaced_count, best_score
# result["history"] — per-iteration list for convergence plotting
```

### Complexity
`O(n_ants × n²)` per iteration. Bottleneck is `pack_boxes()` running `n_ants` times. Scales well to 100+ boxes.

### Strengths / Weaknesses
**Strengths:** positive feedback loop, no gradient needed, natural fit for ordering problems, zero extra dependencies.  
**Weaknesses:** O(n²) pheromone matrix, sensitive to η quality, can stagnate without `tau_min`/`tau_max` tuning.


In [13]:

# ─────────────────────────────────────────────────────────────────────────────
# Helpers
# ─────────────────────────────────────────────────────────────────────────────

def _score(result, w_util=0.7, w_stab=0.3):
    """Unified fitness score identical to the one used in Greedy / GA / SA."""
    return (
        w_util * result["utilization"]
        + w_stab * result["stability"]["stability_score"]
    )


def _heuristic(box):
    """
    Desirability η for placing a box early.
    Larger, heavier boxes should go first → higher η.
    Fragile boxes get a small penalty so they tend to go later.
    """
    volume = box["l"] * box["w"] * box["h"]
    weight = box["weight"]
    fragile_penalty = 0.5 if box["fragile"] else 1.0
    return (volume * weight) ** 0.5 * fragile_penalty


# ─────────────────────────────────────────────────────────────────────────────
# ACOPacker
# ─────────────────────────────────────────────────────────────────────────────

class ACOPacker:
    """
    Ant Colony Optimization for 3D bin packing.

    The problem is modelled as: find the best *ordering* of boxes to
    feed into the container's placement heuristic (BFF).

    Graph structure
    ---------------
    • Nodes  : boxes  (indexed 0 … n-1)
    • Edges  : (i → j) means "box j is placed immediately after box i"
    • τ[i][j]: pheromone on edge i→j
    • η[j]   : heuristic desirability of box j (volume × weight)

    At each construction step an ant at node i picks the next unvisited
    node j with probability:

        P(j | i) = (τ[i][j]^α × η[j]^β) / Σ_k (τ[i][k]^α × η[k]^β)

    After all ants finish their tours the pheromone matrix is updated:

        τ[i][j] ← (1 - ρ) × τ[i][j]  +  Σ_ant  ΔQ / score_ant

    Parameters
    ----------
    container       : Container  – your existing Container instance
    boxes           : list[dict] – box list from load_random_boxes()
    n_ants          : int        – number of ants per iteration (20-50)
    n_iter          : int        – number of iterations
    alpha           : float      – pheromone exponent  (≥1 → trust pheromone more)
    beta            : float      – heuristic exponent  (≥1 → trust heuristic more)
    rho             : float      – evaporation rate in (0, 1)
    Q               : float      – pheromone deposit constant
    tau_init        : float      – initial pheromone value on all edges
    tau_min         : float      – minimum pheromone (prevents stagnation)
    tau_max         : float|None – maximum pheromone (None = unbounded)
    w_util          : float      – utilization weight in fitness
    w_stab          : float      – stability weight in fitness
    seed            : int|None   – random seed
    verbose         : bool       – print progress each iteration
    """

    def __init__(
        self,
        container,
        boxes,
        n_ants    = 30,
        n_iter    = 100,
        alpha     = 1.0,
        beta      = 2.0,
        rho       = 0.1,
        Q         = 100.0,
        tau_init  = 1.0,
        tau_min   = 0.01,
        tau_max   = None,
        w_util    = 0.7,
        w_stab    = 0.3,
        seed      = None,
        verbose   = True,
    ):
        self.container = container
        self.boxes     = boxes
        self.n         = len(boxes)
        self.n_ants    = n_ants
        self.n_iter    = n_iter
        self.alpha     = alpha
        self.beta      = beta
        self.rho       = rho
        self.Q         = Q
        self.tau_min   = tau_min
        self.tau_max   = tau_max
        self.w_util    = w_util
        self.w_stab    = w_stab
        self.verbose   = verbose

        if seed is not None:
            random.seed(seed)

        # ── pheromone matrix  τ[i][j]  (n+1 × n) ───────────────────────────
        # Row n is the virtual "start" node (before any box is placed).
        self.tau = [
            [tau_init] * self.n
            for _ in range(self.n + 1)
        ]

        # ── heuristic desirability η[j] (static, computed once) ─────────────
        self.eta = [_heuristic(b) for b in boxes]

        # ── tracking ─────────────────────────────────────────────────────────
        self.best_order  = None
        self.best_score  = float("-inf")
        self.best_result = None
        self.history     = []   # one entry per iteration

    # ── construction: one ant builds one complete ordering ────────────────────

    def _build_tour(self):
        """
        One ant constructs a full permutation of box indices using the
        ACO probabilistic selection rule.

        Returns
        -------
        list[int]  – permutation of box indices 0…n-1
        """
        unvisited = list(range(self.n))
        tour      = []
        current   = self.n   # start node (virtual)

        while unvisited:
            # ── compute selection weights ────────────────────────────────────
            weights = []
            for j in unvisited:
                tau_ij = self.tau[current][j] ** self.alpha
                eta_j  = self.eta[j]          ** self.beta
                weights.append(tau_ij * eta_j)

            total = sum(weights)

            # ── roulette-wheel selection ─────────────────────────────────────
            r        = random.random() * total
            cumul    = 0.0
            chosen_j = unvisited[-1]   # fallback (floating point safety)

            for idx, j in enumerate(unvisited):
                cumul += weights[idx]
                if cumul >= r:
                    chosen_j = j
                    break

            tour.append(chosen_j)
            unvisited.remove(chosen_j)
            current = chosen_j

        return tour

    # ── evaluation: pack boxes in the ant's order, return score ──────────────

    def _evaluate_tour(self, tour):
        """
        Pack boxes in the order given by *tour* (list of indices).
        Returns (score, result_dict).
        """
        ordered_boxes = [self.boxes[i] for i in tour]
        result        = self.container.pack_boxes(ordered_boxes)
        score         = _score(result, self.w_util, self.w_stab)
        return score, result

    # ── pheromone update ──────────────────────────────────────────────────────

    def _update_pheromone(self, ant_tours, ant_scores):
        """
        1. Evaporate all pheromone trails.
        2. Each ant deposits Δτ = Q / score along the edges it used.
        3. Clamp to [tau_min, tau_max].
        """
        n = self.n

        # ── evaporation ──────────────────────────────────────────────────────
        for i in range(n + 1):
            for j in range(n):
                self.tau[i][j] *= (1.0 - self.rho)
                self.tau[i][j]  = max(self.tau[i][j], self.tau_min)

        # ── deposit ──────────────────────────────────────────────────────────
        for tour, score in zip(ant_tours, ant_scores):
            if score <= 0:
                continue
            delta = self.Q / score

            # virtual start node → first box
            self.tau[n][tour[0]] += delta

            # box-to-box edges
            for k in range(len(tour) - 1):
                i = tour[k]
                j = tour[k + 1]
                self.tau[i][j] += delta

            # optional cap
            if self.tau_max is not None:
                for i in range(n + 1):
                    for j in range(n):
                        self.tau[i][j] = min(self.tau[i][j], self.tau_max)

    # ── main loop ─────────────────────────────────────────────────────────────

    def run(self):
        """
        Run the ACO algorithm.

        Returns
        -------
        dict with keys:
            best_order   – list of boxes in the best order found
            best_score   – fitness score of the best solution
            stats        – placed_count, unplaced_count, utilization, stability
            history      – list of per-iteration best scores
            elapsed_sec  – wall-clock time
        """
        t0 = time.time()

        for iteration in range(1, self.n_iter + 1):

            ant_tours  = []
            ant_scores = []

            # ── each ant builds and evaluates its tour ────────────────────────
            for _ in range(self.n_ants):
                tour        = self._build_tour()
                score, result = self._evaluate_tour(tour)

                ant_tours.append(tour)
                ant_scores.append(score)

                # global best update
                if score > self.best_score:
                    self.best_score  = score
                    self.best_order  = [self.boxes[i] for i in tour]
                    self.best_result = copy.deepcopy(result)

            # ── pheromone update ──────────────────────────────────────────────
            self._update_pheromone(ant_tours, ant_scores)

            # ── logging ───────────────────────────────────────────────────────
            iter_best_score = max(ant_scores)
            self.history.append({
                "iteration"     : iteration,
                "iter_best"     : round(iter_best_score, 4),
                "global_best"   : round(self.best_score,  4),
                "utilization"   : round(self.best_result["utilization"], 4),
                "stability"     : round(self.best_result["stability"]["stability_score"], 4),
                "placed_count"  : self.best_result["placed_count"],
                "unplaced_count": self.best_result["unplaced_count"],
            })

            if self.verbose:
                h = self.history[-1]
                print(
                    f"[ACO] iter {iteration:>4}/{self.n_iter}  "
                    f"iter_best={h['iter_best']:>8.4f}  "
                    f"global_best={h['global_best']:>8.4f}  "
                    f"util={h['utilization']:>6.2f}%  "
                    f"stab={h['stability']:>6.2f}  "
                    f"placed={h['placed_count']}/{self.n}"
                )

        elapsed = time.time() - t0

        # ── re-pack with best order to leave container in correct state ───────
        self.container.pack_boxes(self.best_order)

        stats = {
            "best_score"    : round(self.best_score, 4),
            "utilization"   : round(self.best_result["utilization"], 4),
            "stability"     : self.best_result["stability"]["stability_score"],
            "stability_full": self.best_result["stability"],
            "placed_count"  : self.best_result["placed_count"],
            "unplaced_count": self.best_result["unplaced_count"],
            "placed_boxes"  : self.best_result["placed_boxes"],
            "unplaced_boxes": self.best_result["unplaced_boxes"],
        }

        return {
            "best_order" : self.best_order,
            "best_score" : self.best_score,
            "stats"      : stats,
            "history"    : self.history,
            "elapsed_sec": round(elapsed, 2),
        }



In [ ]:


if __name__ == "__main__":

    sample_boxes = [
        {"id": str(i), "name": f"Box{i}", "l": l, "w": w, "h": h,
         "weight": wt, "fragile": fr}
        for i, (l, w, h, wt, fr) in enumerate([
            (100, 80,  60,  10.0, False),
            (120, 90,  70,  15.0, False),
            (60,  50,  40,  5.0,  True ),
            (200, 100, 80,  25.0, False),
            (80,  70,  50,  8.0,  False),
            (150, 120, 90,  20.0, False),
            (50,  40,  30,  3.0,  True ),
            (90,  80,  60,  12.0, False),
            (110, 100, 70,  18.0, False),
            (70,  60,  50,  7.0,  False),
        ])
    ]


    container = Container()

    aco = ACOPacker(
        container = container,
        boxes     = sample_boxes,
        n_ants    = 10,
        n_iter    = 20,
        alpha     = 1.0,
        beta      = 2.0,
        rho       = 0.1,
        Q         = 100.0,
        seed      = 42,
        verbose   = True,
    )

    result = aco.run()

    print("\n=== ACO RESULT ===")
    print(f"  Elapsed      : {result['elapsed_sec']}s")
    print(f"  Score        : {result['best_score']:.4f}")
    print(f"  Utilization  : {result['stats']['utilization']}%")
    print(f"  Stability    : {result['stats']['stability']}")
    print(f"  Placed       : {result['stats']['placed_count']} / {len(sample_boxes)}")
    print(f"  Unplaced     : {result['stats']['unplaced_count']}")


[ACO] iter    1/20  iter_best= 42.8334  global_best= 42.8334  util= 19.10%  stab= 98.22  placed=10/10
[ACO] iter    2/20  iter_best= 43.3674  global_best= 43.3674  util= 19.10%  stab=100.00  placed=10/10
[ACO] iter    3/20  iter_best= 43.3674  global_best= 43.3674  util= 19.10%  stab=100.00  placed=10/10
[ACO] iter    4/20  iter_best= 42.5664  global_best= 43.3674  util= 19.10%  stab=100.00  placed=10/10
[ACO] iter    5/20  iter_best= 42.8334  global_best= 43.3674  util= 19.10%  stab=100.00  placed=10/10
[ACO] iter    6/20  iter_best= 42.8334  global_best= 43.3674  util= 19.10%  stab=100.00  placed=10/10
[ACO] iter    7/20  iter_best= 43.1004  global_best= 43.3674  util= 19.10%  stab=100.00  placed=10/10
[ACO] iter    8/20  iter_best= 42.5664  global_best= 43.3674  util= 19.10%  stab=100.00  placed=10/10
[ACO] iter    9/20  iter_best= 42.8334  global_best= 43.3674  util= 19.10%  stab=100.00  placed=10/10
[ACO] iter   10/20  iter_best= 43.3674  global_best= 43.3674  util= 19.10%  stab=1


# Constraint Satisfaction (CSP)

### Core Idea
CSP frames the problem as: *find an assignment of (position, orientation) to every box that satisfies all constraints*. Unlike OR-Tools CP-SAT (which requires a solver install and breaks past ~25 boxes), this implementation uses **backtracking search with constraint propagation** directly on your Container's free-space list — no dependencies, scales to 100+ boxes.

### Three Phases

**Phase 1 — Domain Reduction (before search starts)**
For each box, compute all orientations that can physically fit inside the container. Prune the rest. This cuts the search space before a single placement is attempted.

Boxes are then sorted by **Most Constrained Variable (MCV)** ordering: largest volume first (hardest to place → decide early), fragile boxes last.

**Phase 2 — Constraint Checking (at each placement attempt)**

| Constraint | Type | Action on violation |
|---|---|---|
| Geometric fit: dims ≤ free space | Hard | Skip this orientation/space |
| Support ratio ≥ 30% of base area | Hard | Skip this orientation/space |
| Fragile box under non-fragile | Hard | Skip this orientation/space |
| Heavier box above lighter one | Soft | Log warning, allow placement |

**Phase 3 — Backtracking Search**
```
for each box in MCV order:
    for each valid (orientation, free_space) pair:
        check all constraints
        if all pass:
            place box, split free space, recurse to next box
            if recursion succeeds → done
            else → UNDO placement, try next pair   ← backtrack
    if no pair works → leave box unplaced, continue
```
A `backtrack_limit` caps worst-case runtime. When hit, the current partial solution is accepted.

### Parameters

| Parameter | Default | Role |
|---|---|---|
| `backtrack_limit` | 500 | Max backtracks before accepting partial solution |
| `min_support` | 0.30 | Minimum support ratio (30% of base area) |
| `w_util` | 0.7 | Utilization weight in score |
| `w_stab` | 0.3 | Stability weight in score |

**Tuning `backtrack_limit`:**
- `100–300` → fast, greedy-like, fewer placed boxes
- `500–1000` → balanced
- `2000+` → thorough but slow for large instances

### Usage
```python
from container   import Container
from csp_packing import CSPPacker

csp = CSPPacker(
    container=container, boxes=boxes,
    backtrack_limit=500,
    verbose=True,
)
result = csp.run()

# result["stats"] keys:
#   utilization, stability, placed_count, unplaced_count, best_score
# result["backtracks"] — how many backtracks were triggered
```

### Complexity
`O(n × d × b)` where `d` = domain size per box (≤6 orientations) and `b` = number of free spaces. In practice backtracking is rare on 3D packing because the greedy MCV ordering usually finds a valid placement on the first try.

### Strengths / Weaknesses
**Strengths:** enforces hard constraints exactly (support, fragile), no dependencies, backtracking gives it recovery ability that pure greedy lacks, highly interpretable.  
**Weaknesses:** backtrack limit makes it a heuristic in disguise for large instances; solution quality depends heavily on MCV ordering; no global optimization (unlike ACO/GA/SA).

---

## Plugging Both Into `test_diff()`

```python
# load separate box lists
boxes5 = load_random_boxes("../data/data.csv", n=100, seed=seed)
boxes6 = load_random_boxes("../data/data.csv", n=100, seed=seed)

# ACO
aco        = ACOPacker(container=container, boxes=boxes5, n_ants=30, n_iter=100, seed=42, verbose=False)
aco_result = aco.run()
aco_stats  = aco_result["stats"]

# CSP
csp        = CSPPacker(container=container, boxes=boxes6, backtrack_limit=500, verbose=False)
csp_result = csp.run()
csp_stats  = csp_result["stats"]

# both expose the same stat keys:
#   stats["placed_count"], stats["unplaced_count"],
#   stats["utilization"],  stats["stability"],  stats["best_score"]
```

## Quick Comparison

| | ACO | CSP |
|---|---|---|
| **Optimises** | Box ordering | Position + orientation per box |
| **Placement engine** | Your `pack_boxes()` | Own backtracking on free-space list |
| **Hard constraints** | None (soft via fitness) | Support ratio, fragile stacking |
| **Scales to 100+ boxes** | ✅ | ✅ |
| **Extra dependencies** | None | None |
| **Best when** | You want learned, improving orderings | You need constraint guarantees |


In [14]:

# ─────────────────────────────────────────────────────────────────────────────
# Helpers
# ─────────────────────────────────────────────────────────────────────────────

def _get_orientations(box, container_L, container_W, container_H):
    """
    All valid orientations for a box, pre-filtered to those that can
    physically fit inside the container in at least one axis alignment.
    Fragile boxes keep h fixed (only l and w may swap).
    """
    l, w, h = box["l"], box["w"], box["h"]
    if box["fragile"]:
        candidates = list(set([(l, w, h), (w, l, h)]))
    else:
        candidates = list(set(permutations((l, w, h))))

    # prune orientations that cannot fit the container at all
    return [
        (bl, bw, bh) for (bl, bw, bh) in candidates
        if bl <= container_L and bw <= container_W and bh <= container_H
    ]


def _support_ratio(placed_boxes, box_x, box_y, box_z, bl, bw):
    """
    Fraction of the box's base area that is supported by boxes directly below.
    Returns 1.0 if the box is on the floor (z == 0).
    """
    if box_z == 0:
        return 1.0

    base_area    = bl * bw
    supported    = 0.0
    box_x2, box_y2 = box_x + bl, box_y + bw

    for other in placed_boxes:
        if other["z"] + other["h"] != box_z:
            continue
        ox1, ox2 = other["x"], other["x"] + other["l"]
        oy1, oy2 = other["y"], other["y"] + other["w"]
        overlap_x = max(0, min(box_x2, ox2) - max(box_x, ox1))
        overlap_y = max(0, min(box_y2, oy2) - max(box_y, oy1))
        supported += overlap_x * overlap_y

    return supported / base_area if base_area > 0 else 0.0


def _score(result, w_util=0.7, w_stab=0.3):
    return (
        w_util * result["utilization"]
        + w_stab * result["stability"]["stability_score"]
    )


# ─────────────────────────────────────────────────────────────────────────────
# CSPPacker
# ─────────────────────────────────────────────────────────────────────────────

class CSPPacker:
    """
    Constraint Satisfaction Packer for 3D bin packing.

    Parameters
    ----------
    container        : Container  – your existing Container instance
    boxes            : list[dict] – box list from load_random_boxes()
    backtrack_limit  : int        – max backtracks before giving up (default 500)
    min_support      : float      – minimum support ratio required (default 0.30)
    w_util           : float      – utilization weight in score
    w_stab           : float      – stability weight in score
    verbose          : bool       – print progress
    """

    def __init__(
        self,
        container,
        boxes,
        backtrack_limit = 500,
        min_support     = 0.30,
        w_util          = 0.7,
        w_stab          = 0.3,
        verbose         = True,
    ):
        self.container       = container
        self.original_boxes  = boxes
        self.backtrack_limit = backtrack_limit
        self.min_support     = min_support
        self.w_util          = w_util
        self.w_stab          = w_stab
        self.verbose         = verbose

        self.L = container.length
        self.W = container.width
        self.H = container.height

        self._backtracks = 0

    # ── PHASE 1: preprocessing ────────────────────────────────────────────────

    def _preprocess(self):
        """
        1. Prune impossible orientations per box.
        2. Sort boxes: Most Constrained Variable (MCV) first.
           MCV = largest volume first (harder to place → decide early),
           fragile boxes pushed toward the end.
        """
        boxes_with_domains = []
        for box in self.original_boxes:
            domain = _get_orientations(box, self.L, self.W, self.H)
            if domain:   # boxes with zero valid orientations are unplaceable
                boxes_with_domains.append((box, domain))
            else:
                if self.verbose:
                    print(f"[CSP] Box {box['id']} pruned — no valid orientation fits the container.")

        # MCV ordering: largest volume first, fragile last
        boxes_with_domains.sort(
            key=lambda bd: (
                bd[0]["fragile"],                           # False < True → non-fragile first
                -(bd[0]["l"] * bd[0]["w"] * bd[0]["h"]),   # larger volume first
            )
        )

        return boxes_with_domains

    # ── PHASE 2: constraint checks ────────────────────────────────────────────

    def _check_weight_constraint(self, placed_boxes, new_z, new_weight):
        """
        Soft weight constraint: warn if a heavier box is being placed above
        a lighter one. Returns True always (soft — does not block placement).
        """
        for other in placed_boxes:
            if other["weight"] > new_weight and other["z"] > new_z:
                return False   # violation detected (caller logs it, doesn't block)
        return True

    def _check_fragile_constraint(self, placed_boxes, new_box_id, new_x, new_y,
                                   new_z, bl, bw, bh, fragile):
        """
        Hard fragile constraint: a fragile box must not have any non-fragile
        box placed on top of it. We check prospectively: if placing this box
        at (x,y,z) with dims (bl,bw,bh) would sit ON TOP of a fragile box,
        reject it.
        """
        if fragile:
            return True   # fragile box placing on anything is fine for itself

        # this is a non-fragile box — check it's not sitting on a fragile one
        for other in placed_boxes:
            if not other.get("fragile", False):
                continue
            # other is fragile; check if new box sits directly on top of it
            other_top = other["z"] + other["h"]
            if other_top != new_z:
                continue
            # check XY overlap
            ox1, ox2 = other["x"], other["x"] + other["l"]
            oy1, oy2 = other["y"], other["y"] + other["w"]
            overlap_x = max(0, min(new_x + bl, ox2) - max(new_x, ox1))
            overlap_y = max(0, min(new_y + bw, oy2) - max(new_y, oy1))
            if overlap_x > 0 and overlap_y > 0:
                return False   # non-fragile sitting on fragile → reject

        return True

    # ── PHASE 3: backtracking search ─────────────────────────────────────────

    def _backtrack(self, boxes_with_domains, index, state):
        """
        Recursive backtracking search.

        state = {
            "placed_boxes" : list of placed box dicts,
            "free_spaces"  : list of current free spaces,
            "unplaced_ids" : set of box ids not yet placed,
        }

        Returns True when all boxes have been attempted (success or limit hit).
        """
        if self._backtracks > self.backtrack_limit:
            return True   # cut off — accept current partial solution

        if index == len(boxes_with_domains):
            return True   # all boxes processed

        box, domain = boxes_with_domains[index]

        # try every (orientation, free_space) combination
        for (bl, bw, bh) in domain:
            for space in list(state["free_spaces"]):

                # ── geometric fit ────────────────────────────────────────────
                if bl > space["L"] or bw > space["W"] or bh > space["H"]:
                    continue

                x, y, z = space["x"], space["y"], space["z"]

                # ── support constraint ───────────────────────────────────────
                ratio = _support_ratio(state["placed_boxes"], x, y, z, bl, bw)
                if ratio < self.min_support:
                    continue

                # ── fragile constraint (hard) ────────────────────────────────
                if not self._check_fragile_constraint(
                    state["placed_boxes"], box["id"], x, y, z, bl, bw, bh, box["fragile"]
                ):
                    continue

                # ── weight constraint (soft — just log) ──────────────────────
                weight_ok = self._check_weight_constraint(
                    state["placed_boxes"], z, box["weight"]
                )
                if self.verbose and not weight_ok:
                    print(f"[CSP] Weight soft-violation: box {box['id']} (w={box['weight']}) placed above a heavier box.")

                # ── PLACE the box ─────────────────────────────────────────────
                placed_entry = {
                    "id"      : box["id"],
                    "name"    : box["name"],
                    "weight"  : box["weight"],
                    "fragile" : box["fragile"],
                    "x": x, "y": y, "z": z,
                    "l": bl,  "w": bw,  "h": bh,
                }

                # save state for backtracking
                old_free_spaces   = copy.deepcopy(state["free_spaces"])
                old_placed_boxes  = copy.deepcopy(state["placed_boxes"])

                state["placed_boxes"].append(placed_entry)
                state["free_spaces"].remove(space)
                new_spaces = self._split_space(space, bl, bw, bh, x, y, z)
                state["free_spaces"].extend(new_spaces)

                # ── recurse ───────────────────────────────────────────────────
                if self._backtrack(boxes_with_domains, index + 1, state):
                    return True

                # ── BACKTRACK ─────────────────────────────────────────────────
                self._backtracks += 1
                state["placed_boxes"] = old_placed_boxes
                state["free_spaces"]  = old_free_spaces

                if self.verbose:
                    print(f"[CSP] Backtrack #{self._backtracks} — box {box['id']}")

        # no valid placement found for this box → leave it unplaced, continue
        # (partial placement is still a valid outcome)
        return self._backtrack(boxes_with_domains, index + 1, state)

    def _split_space(self, space, l, w, h, x, y, z):
        """Identical split logic to Container.split_space()."""
        new_spaces = []
        if space["L"] > l:
            new_spaces.append({"x": x+l, "y": y,   "z": z,   "L": space["L"]-l, "W": space["W"],   "H": space["H"]})
        if space["W"] > w:
            new_spaces.append({"x": x,   "y": y+w,  "z": z,   "L": l,            "W": space["W"]-w, "H": space["H"]})
        if space["H"] > h:
            new_spaces.append({"x": x,   "y": y,    "z": z+h, "L": l,            "W": w,            "H": space["H"]-h})
        return new_spaces

    # ── main entry point ──────────────────────────────────────────────────────

    def run(self):
        """
        Run the CSP solver.

        Returns
        -------
        dict with keys:
            stats        – placed_count, unplaced_count, utilization,
                           stability (float), stability_full (dict),
                           placed_boxes, unplaced_boxes, best_score
            backtracks   – number of backtracks performed
            elapsed_sec  – wall-clock time
        """
        t0 = time.time()

        # ── phase 1: preprocess ───────────────────────────────────────────────
        boxes_with_domains = self._preprocess()

        if self.verbose:
            print(f"[CSP] {len(boxes_with_domains)} boxes with valid domains, "
                  f"backtrack limit = {self.backtrack_limit}")

        # ── phase 2 + 3: search ───────────────────────────────────────────────
        state = {
            "placed_boxes": [],
            "free_spaces" : [{"x":0,"y":0,"z":0,
                               "L":self.L,"W":self.W,"H":self.H}],
        }

        self._backtracks = 0
        self._backtrack(boxes_with_domains, 0, state)

        # ── collect unplaced boxes ────────────────────────────────────────────
        placed_ids  = {b["id"] for b in state["placed_boxes"]}
        unplaced    = [
            {"id": b["id"], "reason": "not placed by CSP search"}
            for b in self.original_boxes
            if b["id"] not in placed_ids
        ]

        # ── compute statistics via Container ─────────────────────────────────
        self.container.placed_boxes   = state["placed_boxes"]
        self.container.unplaced_boxes = unplaced
        self.container.free_spaces    = state["free_spaces"]

        packed_volume    = sum(b["l"]*b["w"]*b["h"] for b in state["placed_boxes"])
        container_volume = self.L * self.W * self.H
        utilization      = packed_volume / container_volume * 100
        stability        = self.container.evaluate_stability()
        score            = _score({"utilization": utilization, "stability": stability},
                                   self.w_util, self.w_stab)

        elapsed = time.time() - t0

        if self.verbose:
            print(f"\n[CSP] Done in {elapsed:.2f}s | backtracks={self._backtracks}")
            print(f"[CSP] Placed={len(state['placed_boxes'])} | "
                  f"Utilization={utilization:.2f}% | "
                  f"Stability={stability['stability_score']:.2f} | "
                  f"Score={score:.4f}")

        stats = {
            "best_score"    : round(score, 4),
            "utilization"   : round(utilization, 4),
            "stability"     : stability["stability_score"],
            "stability_full": stability,
            "placed_count"  : len(state["placed_boxes"]),
            "unplaced_count": len(unplaced),
            "placed_boxes"  : state["placed_boxes"],
            "unplaced_boxes": unplaced,
        }

        return {
            "stats"      : stats,
            "backtracks" : self._backtracks,
            "elapsed_sec": round(elapsed, 2),
        }




In [ ]:

if __name__ == "__main__":

    sample_boxes = [
        {"id": str(i), "name": f"Box{i}", "l": l, "w": w, "h": h,
         "weight": wt, "fragile": fr}
        for i, (l, w, h, wt, fr) in enumerate([
            (100, 80,  60,  10.0, False),
            (120, 90,  70,  15.0, False),
            (60,  50,  40,  5.0,  True ),
            (200, 100, 80,  25.0, False),
            (80,  70,  50,  8.0,  False),
            (150, 120, 90,  20.0, False),
            (50,  40,  30,  3.0,  True ),
            (90,  80,  60,  12.0, False),
            (110, 100, 70,  18.0, False),
            (70,  60,  50,  7.0,  False),
        ])
    ]


    container = Container()
    csp = CSPPacker(
        container       = container,
        boxes           = sample_boxes,
        backtrack_limit = 200,
        verbose         = True,
    )

    result = csp.run()
    print("\n=== CSP RESULT ===")
    s = result["stats"]
    print(f"  Score        : {s['best_score']}")
    print(f"  Utilization  : {s['utilization']}%")
    print(f"  Stability    : {s['stability']}")
    print(f"  Placed       : {s['placed_count']} / {len(sample_boxes)}")
    print(f"  Backtracks   : {result['backtracks']}")
    print(f"  Elapsed      : {result['elapsed_sec']}s")


[CSP] 10 boxes with valid domains, backtrack limit = 200
[CSP] Weight soft-violation: box 1 (w=15.0) placed above a heavier box.
[CSP] Weight soft-violation: box 0 (w=10.0) placed above a heavier box.
[CSP] Weight soft-violation: box 7 (w=12.0) placed above a heavier box.
[CSP] Weight soft-violation: box 4 (w=8.0) placed above a heavier box.
[CSP] Weight soft-violation: box 9 (w=7.0) placed above a heavier box.
[CSP] Weight soft-violation: box 2 (w=5.0) placed above a heavier box.
[CSP] Weight soft-violation: box 6 (w=3.0) placed above a heavier box.

[CSP] Done in 0.00s | backtracks=0
[CSP] Placed=10 | Utilization=19.10% | Stability=91.11 | Score=40.7004

=== CSP RESULT ===
  Score        : 40.7004
  Utilization  : 19.0963%
  Stability    : 91.11
  Placed       : 10 / 10
  Backtracks   : 0
  Elapsed      : 0.0s


# This Section is for Comparison Between All Algorithms:

- Genetic Algorithm
- Simulated Anealling
- Greedy
- Particle Swarm Optimization 

In [15]:
def test_diff():
    container = Container()

    seed  = random.randint(1, 1_000_000)
    boxes = load_random_boxes("../data/data.csv", n=100, seed=seed)

    print("=" * 70)
    print(f"  SHARED SETUP  —  n={len(boxes)} boxes  |  seed={seed}")
    print("=" * 70)
    print(f"  Same {len(boxes)} boxes passed to all 4 searches.")
    print()

    # ══════════════════════════════════════════════════════════════════════════
    # 1. GREEDY BFD
    # ══════════════════════════════════════════════════════════════════════════
    print("=" * 70)
    print("  RUNNING: Greedy BFD")
    print("=" * 70)
    print("  Strategy  : Sort boxes by decreasing volume, place each in best-fit space")
    print("  Parameters: no tunable parameters — deterministic single pass")
    print("  Running...")

    bfd        = GreedyBFD(container, boxes)
    bfd_result = bfd.run()
    bfd_r      = bfd_result["best_result"]

    print()
    print("  ── BFD RESULT ───────────────────────────────────────────────────")
    print(f"  Score          : {bfd_result['best_score']:.4f}")
    print(f"  Placed boxes   : {bfd_r['placed']} / {len(boxes)}")
    print(f"  Unplaced boxes : {bfd_r['unplaced']}")
    print(f"  Utilisation    : {bfd_r['utilization']:.2f} %")
    print(f"  Stability      : {bfd_r['stability']['stability_score']:.2f}")
    print(f"  Weight score   : {bfd_r['stability']['weight_score']:.2f}")
    print(f"  Support score  : {bfd_r['stability']['support_score']:.2f}")
    print()

    # ══════════════════════════════════════════════════════════════════════════
    # 2. SIMULATED ANNEALING
    # ══════════════════════════════════════════════════════════════════════════
    print("=" * 70)
    print("  RUNNING: Simulated Annealing")
    print("=" * 70)
    print("  Strategy  : Random swaps accepted probabilistically based on temperature")
    print("  Parameters:")
    print(f"    T (initial temperature) = 5000")
    print(f"    alpha (cooling rate)    = 0.98")
    print(f"    iterations              = 500")
    print(f"    w_util / w_stab / w_place = 0.4 / 0.3 / 0.3")
    print("  Running...")

    sa        = SimulatedAnnealing(container=container, intitial_boxes=boxes)
    sa_r      = sa.run(500)
    sa_result = sa_r["best_result"]
    sa_score  = sa_r["best_score"]

    print()
    print("  ── SA RESULT ────────────────────────────────────────────────────")
    print(f"  Score          : {sa_score:.4f}")
    print(f"  Placed boxes   : {sa_result['placed_count']} / {len(boxes)}")
    print(f"  Unplaced boxes : {sa_result['unplaced_count']}")
    print(f"  Utilisation    : {sa_result['utilization']:.2f} %")
    print(f"  Stability      : {sa_result['stability']['stability_score']:.2f}")
    print(f"  Weight score   : {sa_result['stability']['weight_score']:.2f}")
    print(f"  Support score  : {sa_result['stability']['support_score']:.2f}")
    print()

    # ══════════════════════════════════════════════════════════════════════════
    # 3. GENETIC ALGORITHM
    # ══════════════════════════════════════════════════════════════════════════
    print("=" * 70)
    print("  RUNNING: Genetic Algorithm")
    print("=" * 70)
    print("  Strategy  : Evolve population of box orderings via selection, crossover, mutation")

    ga = GeneticAlgorithm(container=container, boxes=boxes, max_evals=10_000)

    print(f"    w_util / w_stab / w_place = 0.4 / 0.3 / 0.3")
    print("  Running...")

    ga_r      = ga.run()
    ga_result = ga_r["best_result"]
    ga_score  = ga_r["best_score"]

    print()
    print("  ── GA RESULT ────────────────────────────────────────────────────")
    print(f"  Score          : {ga_score:.4f}")
    print(f"  Placed boxes   : {ga_result['placed_count']} / {len(boxes)}")
    print(f"  Unplaced boxes : {ga_result['unplaced_count']}")
    print(f"  Utilisation    : {ga_result['utilization']:.2f} %")
    print(f"  Stability      : {ga_result['stability']['stability_score']:.2f}")
    print(f"  Weight score   : {ga_result['stability']['weight_score']:.2f}")
    print(f"  Support score  : {ga_result['stability']['support_score']:.2f}")
    print()

    # ══════════════════════════════════════════════════════════════════════════
    # 4. PSO
    # ══════════════════════════════════════════════════════════════════════════
    print("=" * 70)
    print("  RUNNING: Particle Swarm Optimisation")
    print("=" * 70)
    print("  Strategy  : Swarm of particles explore box orderings guided by personal and global best")
    print("  Parameters:")
    print(f"    n_particles      = 100")
    print(f"    n_iter           = 50")
    print(f"    w (inertia)      = 0.7")
    print(f"    c1 (cognitive)   = 1.5")
    print(f"    c2 (social)      = 1.5")
    print(f"    lambda_stability = 0.3")
    print(f"    lambda_unplaced  = 5.0")
    print(f"    seed             = {seed}")
    print("  Running...")

    pso = PSOPacker(
        container        = container,
        boxes            = boxes,
        n_particles      = 100,
        n_iter           = 50,
        w                = 0.7,
        c1               = 1.5,
        c2               = 1.5,
        lambda_stability = 0.3,
        lambda_unplaced  = 5.0,
        seed             = seed,
        verbose          = True,
    )
    pso_result = pso.run()
    pso_stats  = pso_result["stats"]

    pso_placement_rate = (pso_stats["placed_count"] / len(boxes)) * 100
    pso_score = (
        0.4 * pso_stats["utilization"]
        + 0.3 * pso_stats["stability"]
        + 0.3 * pso_placement_rate
    )

    print()
    print("  ── PSO RESULT ───────────────────────────────────────────────────")
    print(f"  Score          : {pso_score:.4f}")
    print(f"  Placed boxes   : {pso_stats['placed_count']} / {len(boxes)}")
    print(f"  Unplaced boxes : {pso_stats['unplaced_count']}")
    print(f"  Utilisation    : {pso_stats['utilization']:.2f} %")
    print(f"  Stability      : {pso_stats['stability']:.2f}")
    print(f"  Elapsed        : {pso_result['elapsed_sec']}s")
    print()

    # ══════════════════════════════════════════════════════════════════════════
    # SCORES
    # ══════════════════════════════════════════════════════════════════════════
    bfd_score = bfd_result["best_score"]


    # 5. ACO
    # ══════════════════════════════════════════════════════════════════════════
    print("=" * 70)
    print("  RUNNING: Ant Colony Optimisation")
    print("=" * 70)
    print("  Strategy  : Ants build box orderings guided by pheromone trails and heuristic desirability")
    print("  Parameters:")
    print(f"    n_ants           = 30")
    print(f"    n_iter           = 100")
    print(f"    alpha (pheromone)= 1.0")
    print(f"    beta  (heuristic)= 2.0")
    print(f"    rho   (evaporat.)= 0.1")
    print(f"    Q                = 100.0")
    print(f"    seed             = {seed}")
    print("  Running...")
 
    aco = ACOPacker(
        container = container,
        boxes     = boxes,
        n_ants    = 30,
        n_iter    = 100,
        alpha     = 1.0,
        beta      = 2.0,
        rho       = 0.1,
        Q         = 100.0,
        seed      = seed,
        verbose   = False,
    )
    aco_result       = aco.run()
    aco_stats        = aco_result["stats"]
    aco_placement_rate = (aco_stats["placed_count"] / len(boxes)) * 100
    aco_score = (
        0.4 * aco_stats["utilization"]
        + 0.3 * aco_stats["stability"]
        + 0.3 * aco_placement_rate
    )
 
    print()
    print("  ── ACO RESULT ───────────────────────────────────────────────────")
    print(f"  Score          : {aco_score:.4f}")
    print(f"  Placed boxes   : {aco_stats['placed_count']} / {len(boxes)}")
    print(f"  Unplaced boxes : {aco_stats['unplaced_count']}")
    print(f"  Utilisation    : {aco_stats['utilization']:.2f} %")
    print(f"  Stability      : {aco_stats['stability']:.2f}")
    print(f"  Elapsed        : {aco_result['elapsed_sec']}s")
    print()
 
    # ══════════════════════════════════════════════════════════════════════════
    # 6. CSP
    # ══════════════════════════════════════════════════════════════════════════
    print("=" * 70)
    print("  RUNNING: Constraint Satisfaction")
    print("=" * 70)
    print("  Strategy  : Backtracking search with hard constraint propagation (support, fragile)")
    print("  Parameters:")
    print(f"    backtrack_limit  = 500")
    print(f"    min_support      = 0.30")
    print(f"    MCV ordering     : largest volume first, fragile last")
    print("  Running...")
 
    csp = CSPPacker(
        container       = container,
        boxes           = boxes,
        backtrack_limit = 500,
        verbose         = False,
    )
    csp_result       = csp.run()
    csp_stats        = csp_result["stats"]
    csp_placement_rate = (csp_stats["placed_count"] / len(boxes)) * 100
    csp_score = (
        0.4 * csp_stats["utilization"]
        + 0.3 * csp_stats["stability"]
        + 0.3 * csp_placement_rate
    )
 
    print()
    print("  ── CSP RESULT ───────────────────────────────────────────────────")
    print(f"  Score          : {csp_score:.4f}")
    print(f"  Placed boxes   : {csp_stats['placed_count']} / {len(boxes)}")
    print(f"  Unplaced boxes : {csp_stats['unplaced_count']}")
    print(f"  Utilisation    : {csp_stats['utilization']:.2f} %")
    print(f"  Stability      : {csp_stats['stability']:.2f}")
    print(f"  Backtracks     : {csp_result['backtracks']}")
    print(f"  Elapsed        : {csp_result['elapsed_sec']}s")
    print()

    # ══════════════════════════════════════════════════════════════════════════
    # COMPARISON TABLE
    # ══════════════════════════════════════════════════════════════════════════
    print("\n")
    print("=" * 90)
    print(f"        FINAL COMPARISON  —  n={len(boxes)} boxes  |  seed={seed}")
    print("=" * 90)
    print(f"{'Metric':<26} {'SA':>10} {'Greedy BFD':>12} {'GA':>10} {'PSO':>10} {'ACO':>10} {'CSP':>10}")
    print("-" * 90)
    print(f"{'Score':<26} {sa_score:>10.4f} {bfd_score:>12.4f} {ga_score:>10.4f} {pso_score:>10.4f} {aco_score:>10.4f} {csp_score:>10.4f}")
    print(f"{'Placed boxes':<26} {sa_result['placed_count']:>10} {bfd_r['placed']:>12} {ga_result['placed_count']:>10} {pso_stats['placed_count']:>10} {aco_stats['placed_count']:>10} {csp_stats['placed_count']:>10}")
    print(f"{'Unplaced boxes':<26} {sa_result['unplaced_count']:>10} {bfd_r['unplaced']:>12} {ga_result['unplaced_count']:>10} {pso_stats['unplaced_count']:>10} {aco_stats['unplaced_count']:>10} {csp_stats['unplaced_count']:>10}")
    print(f"{'Utilisation (%)':<26} {sa_result['utilization']:>10.2f} {bfd_r['utilization']:>12.2f} {ga_result['utilization']:>10.2f} {pso_stats['utilization']:>10.2f} {aco_stats['utilization']:>10.2f} {csp_stats['utilization']:>10.2f}")
    print(f"{'Stability score':<26} {sa_result['stability']['stability_score']:>10.2f} {bfd_r['stability']['stability_score']:>12.2f} {ga_result['stability']['stability_score']:>10.2f} {pso_stats['stability']:>10.2f} {aco_stats['stability']:>10.2f} {csp_stats['stability']:>10.2f}")
    print(f"{'Weight score':<26} {sa_result['stability']['weight_score']:>10.2f} {bfd_r['stability']['weight_score']:>12.2f} {ga_result['stability']['weight_score']:>10.2f} {'N/A':>10} {'N/A':>10} {csp_stats['stability_full']['weight_score']:>10.2f}")
    print(f"{'Support score':<26} {sa_result['stability']['support_score']:>10.2f} {bfd_r['stability']['support_score']:>12.2f} {ga_result['stability']['support_score']:>10.2f} {'N/A':>10} {'N/A':>10} {csp_stats['stability_full']['support_score']:>10.2f}")
    print("=" * 90)

     # ══════════════════════════════════════════════════════════════════════════
    # WINNER + WHY  (replace your existing one)
    # ══════════════════════════════════════════════════════════════════════════
    scores = {
        "Simulated Annealing": sa_score,
        "Greedy BFD":          bfd_score,
        "Genetic Algorithm":   ga_score,
        "PSO":                 pso_score,
        "ACO":                 aco_score,
        "CSP":                 csp_score,
    }
    winner     = max(scores, key=scores.get)
    sorted_alg = sorted(scores.items(), key=lambda x: x[1], reverse=True)
 
    print(f"\n  WINNER: {winner}  (score: {scores[winner]:.4f})")
    print()
    print("  RANKING:")
    for rank, (name, score) in enumerate(sorted_alg, 1):
        print(f"    {rank}. {name:<25} score = {score:.4f}")
    print()
    print("  WHY THE WINNER WON:")
 
    w_utils = {
        "Simulated Annealing": sa_result["utilization"],
        "Greedy BFD":          bfd_r["utilization"],
        "Genetic Algorithm":   ga_result["utilization"],
        "PSO":                 pso_stats["utilization"],
        "ACO":                 aco_stats["utilization"],
        "CSP":                 csp_stats["utilization"],
    }
    w_scores = {
        "Simulated Annealing": sa_result["stability"]["stability_score"],
        "Greedy BFD":          bfd_r["stability"]["stability_score"],
        "Genetic Algorithm":   ga_result["stability"]["stability_score"],
        "PSO":                 pso_stats["stability"],
        "ACO":                 aco_stats["stability"],
        "CSP":                 csp_stats["stability"],
    }
    w_placed = {
        "Simulated Annealing": sa_result["placed_count"],
        "Greedy BFD":          bfd_r["placed"],
        "Genetic Algorithm":   ga_result["placed_count"],
        "PSO":                 pso_stats["placed_count"],
        "ACO":                 aco_stats["placed_count"],
        "CSP":                 csp_stats["placed_count"],
    }
 
    best_util  = max(w_utils,  key=w_utils.get)
    best_stab  = max(w_scores, key=w_scores.get)
    best_place = max(w_placed, key=w_placed.get)
 
    print(f"    - Score formula : 0.4 × utilisation + 0.3 × stability + 0.3 × placement rate")
    print(f"    - Best utilisation  : {best_util}  ({w_utils[best_util]:.2f}%)")
    print(f"    - Best stability    : {best_stab}  ({w_scores[best_stab]:.2f})")
    print(f"    - Most boxes placed : {best_place}  ({w_placed[best_place]}/{len(boxes)})")
 
    if best_util == winner and best_stab == winner:
        print(f"    → {winner} dominated on both utilisation AND stability.")
    elif best_util == winner:
        print(f"    → {winner} won mainly due to superior utilisation ({w_utils[winner]:.2f}%).")
    elif best_stab == winner:
        print(f"    → {winner} won mainly due to superior stability ({w_scores[winner]:.2f}).")
    else:
        print(f"    → {winner} achieved the best overall balance across all 3 criteria.")
 
    print("=" * 90)
   


if __name__ == "__main__":
 test_diff()

  SHARED SETUP  —  n=100 boxes  |  seed=383130
  Same 100 boxes passed to all 4 searches.

  RUNNING: Greedy BFD
  Strategy  : Sort boxes by decreasing volume, place each in best-fit space
  Parameters: no tunable parameters — deterministic single pass
  Running...

  ── BFD RESULT ───────────────────────────────────────────────────
  Score          : 81.8149
  Placed boxes   : 66 / 100
  Unplaced boxes : 34
  Utilisation    : 84.15 %
  Stability      : 94.52
  Weight score   : 86.29
  Support score  : 100.00

  RUNNING: Simulated Annealing
  Strategy  : Random swaps accepted probabilistically based on temperature
  Parameters:
    T (initial temperature) = 5000
    alpha (cooling rate)    = 0.98
    iterations              = 500
    w_util / w_stab / w_place = 0.4 / 0.3 / 0.3
  Running...

  ── SA RESULT ────────────────────────────────────────────────────
  Score          : 81.1613
  Placed boxes   : 77 / 100
  Unplaced boxes : 23
  Utilisation    : 72.84 %
  Stability      : 96.42
 